# 02 — U* and U*F clustering

This notebook continues from **01_esom_basics**: we take a trained ESOM and apply
the complete topology-based clustering pipeline:

1. Compute the **P-matrix** (density at each neuron)
2. Compute the **U\*-matrix** (U-matrix weighted by inverse density)
3. Run **U\*F flood clustering** — automatic threshold, no k required
4. Visualise the **cluster label map** and topography
5. Map labels back to raw data and evaluate

All charts are **Altair** (interactive, zoomable).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# FCPS tensors: committed tests/fixtures/fcps.npz (see tests/fixtures/README.md to regenerate).


def load_fcps_fixture(name: str):
    for d in [Path.cwd(), *Path.cwd().parents]:
        p = d / "tests" / "fixtures" / "fcps.npz"
        if p.is_file():
            z = np.load(p)
            return z[f"{name}_data"].copy(), z[f"{name}_cls"].copy()
    raise FileNotFoundError(
        "tests/fixtures/fcps.npz not found — run Jupyter from inside the pyesom repo checkout."
    )


from pyesom import ESOM, UStarFloodClustering, compute_ustar
from pyesom.topology.pmatrix import compute_pmatrix
from pyesom.visualization.cluster import plot_cluster_map
from pyesom.visualization.heatmap import plot_grid_heatmap

## 1  Train ESOM on Hepta

Same FCPS tensor as notebook 01 via **`load_fcps_fixture("hepta")`**. U\*F flood clustering follows Moutarde & Ultsch (2005); default **`n_threshold_steps=100`** matches their §2.4 description (scan thresholds on normalized U\*).

Hepta has seven separated blobs — U\*F should recover structure without fixing **k**.

In [2]:
def zscore(x):
    return (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-9)

data, true_labels = load_fcps_fixture("hepta")
data_z = zscore(data)
print(f"Data: {data.shape}, {len(np.unique(true_labels))} true clusters")

GRID_X, GRID_Y = 32, 36
som = ESOM(GRID_X, GRID_Y, data_z.shape[1], random_seed=42, backend='sompy')
som.fit(data_z, iterations=15_000)
print(f"SOM grid: {GRID_X}×{GRID_Y} = {GRID_X*GRID_Y} neurons")

Data: (212, 3), 7 true clusters


CACHEDIR=/Users/mattijnvanhoek/.matplotlib
Using fontManager instance from /Users/mattijnvanhoek/.matplotlib/fontlist-v390.json


SOM grid: 32×36 = 1152 neurons


## 2  P-matrix (density)

- **Hit map** (`pareto_fraction=None`): each sample contributes 1 to its nearest neuron
- **PDE** (`pareto_fraction=0.2013`): Pareto Density Estimation with per-node hypersphere radius

In [3]:
u = som.u_matrix()
p_hit = compute_pmatrix(som.weights, data_z, pareto_fraction=None)
p_pde = compute_pmatrix(som.weights, data_z, pareto_fraction=0.2013)

print(f"U-matrix : {u.shape},  range [{u.min():.3f}, {u.max():.3f}]")
print(f"P hit    : sum={int(p_hit.sum())} (= n_samples), max hits={int(p_hit.max())}")
print(f"P PDE    : min={p_pde.min():.1f}, max={p_pde.max():.1f}")

alt.hconcat(
    plot_grid_heatmap(u, "U-matrix", scheme="greys", value_name="U"),
    plot_grid_heatmap(p_hit, "P-matrix (hit map)", scheme="blues", value_name="hits"),
    plot_grid_heatmap(p_pde, "P-matrix (PDE)", scheme="blues", value_name="PDE"),
).resolve_scale(color="independent")

U-matrix : (32, 36),  range [0.012, 1.000]
P hit    : sum=212 (= n_samples), max hits=2
P PDE    : min=42.0, max=42.0


alt.HConcatChart(...)

## 3  U\*-matrix

```
ScaleFactor(n) = (P_max - P(n)) / (P_max - P_ref)
U*(n)          = U(n) × clip(ScaleFactor(n), max=3)
```

Dense nodes → ScaleFactor < 1 → U\* suppressed (cluster cores look flatter).  
Sparse nodes → ScaleFactor > 1 → U\* amplified (boundaries stand out more).

In [11]:
ustar = compute_ustar(u, p_hit, scalemax=3.0, median_filter_size=1)
print(f"U*-matrix range : [{ustar.min():.4f}, {ustar.max():.4f}]")

alt.hconcat(
    plot_grid_heatmap(u, "U-matrix", scheme="greys", value_name="U"),
    plot_grid_heatmap(ustar, "U*-matrix", scheme="greys", value_name="U*"),
).properties(title="U vs U* — boundaries amplified, cluster cores suppressed"
).resolve_scale(color="independent")

U*-matrix range : [0.0000, 1.1013]


alt.HConcatChart(...)

### ScaleFactor verification

Dense nodes should have ScaleFactor < 1; sparse boundary nodes ScaleFactor > 1.

In [12]:
p_ref = max(p_hit.mean(), np.median(p_hit))
sf = np.where(u > 0, ustar / np.where(u > 0, u, 1.0), np.nan)
dense_mask = p_hit > p_ref
sparse_mask = p_hit < p_ref

print(f"P_ref (robust mean) : {p_ref:.2f}")
print(f"P_max               : {float(p_hit.max()):.2f}")
print(f"Mean ScaleFactor in dense regions  : {np.nanmean(sf[dense_mask]):.3f}  (< 1 expected)")
print(f"Mean ScaleFactor in sparse regions : {np.nanmean(sf[sparse_mask]):.3f}  (> 1 expected)")

P_ref (robust mean) : 0.18
P_max               : 2.00
Mean ScaleFactor in dense regions  : 0.537  (< 1 expected)
Mean ScaleFactor in sparse regions : 1.101  (> 1 expected)


## 4  U\*F flood clustering

The algorithm scans 100 threshold levels, picks the largest gradient step in the
`regionSize(t)` curve, and grows one flood region per local minimum.  Number of
clusters emerges automatically — no k needed.

In [20]:
clf = UStarFloodClustering(min_cluster_size=4, n_threshold_steps=100)
clf.fit(u, p_hit)

print(f"Auto threshold : {clf.threshold_:.4f}")
print(f"Clusters found : {clf.n_clusters_}  (true k = {len(np.unique(true_labels))})")
print(f"Unassigned nodes : {int((clf.labels_ == -1).sum())} / {clf.labels_.size}  "
      f"({100 * (clf.labels_ == -1).mean():.1f} %)")

Auto threshold : 0.5354
Clusters found : 3  (true k = 7)
Unassigned nodes : 174 / 1152  (15.1 %)


## 5  Cluster map

Two views side by side:
- **Left**: U\*-matrix (greyscale) — topographic landscape
- **Right**: Cluster label grid — each colour is one cluster, grey = unassigned boundary node

In [21]:
ustar_chart = plot_grid_heatmap(
    clf.ustar_, f"U*-matrix (threshold = {clf.threshold_:.3f})",
    scheme="greys", value_name="U*", cell_pixels=9.0,
)
label_chart = plot_cluster_map(
    clf.labels_,
    title=f"Cluster map  (K = {clf.n_clusters_},  grey = unassigned)",
    cell_pixels=9.0,
)

alt.hconcat(ustar_chart, label_chart).resolve_scale(color="independent")

alt.HConcatChart(...)

## 6  Map labels back to raw data

`clf.predict(bmu_indices)` looks up the cluster label of each sample's BMU.
Samples whose BMU is a boundary node stay unassigned (label = -1).

In [17]:
bmus = som.bmu_indices(data_z)
pred_labels = clf.predict(bmus)

assigned_mask = pred_labels >= 0
print(f"Samples assigned   : {assigned_mask.sum()} / {len(pred_labels)}")
print(f"Samples unassigned : {(~assigned_mask).sum()}")
print(f"Cluster sizes: {dict(zip(*np.unique(pred_labels[assigned_mask], return_counts=True)))}")

Samples assigned   : 212 / 212
Samples unassigned : 0
Cluster sizes: {np.int64(0): np.int64(152), np.int64(1): np.int64(30), np.int64(2): np.int64(30)}


## 7  Evaluate — scatter comparison

Side-by-side scatter of features 0 vs 1, coloured by true label (left) and
U\*F predicted label (right).  Grey dots = unassigned samples.

In [18]:
try:
    from sklearn.metrics import adjusted_rand_score
    ari = adjusted_rand_score(true_labels[assigned_mask], pred_labels[assigned_mask])
    print(f"Adjusted Rand Index (assigned samples): {ari:.4f}")
except ImportError:
    print("scikit-learn not installed — skipping ARI")

df_true = pd.DataFrame({
    "x0": data_z[:, 0], "x1": data_z[:, 1],
    "label": true_labels.astype(str),
})
df_pred = pd.DataFrame({
    "x0": data_z[:, 0], "x1": data_z[:, 1],
    "cluster": pred_labels.astype(str),
    "assigned": pred_labels >= 0,
})

true_scatter = (
    alt.Chart(df_true).mark_circle(size=28, opacity=0.75)
    .encode(
        x=alt.X("x0:Q", title="Feature 0 (z-scored)"),
        y=alt.Y("x1:Q", title="Feature 1 (z-scored)"),
        color=alt.Color("label:N", scale=alt.Scale(scheme="tableau10"),
                        legend=alt.Legend(title="True class")),
        tooltip=["x0", "x1", "label"],
    ).properties(title="True labels (Hepta)", width=300, height=280)
)

pred_scatter = alt.layer(
    alt.Chart(df_pred).transform_filter(alt.datum.assigned)
    .mark_circle(size=28, opacity=0.75)
    .encode(
        x=alt.X("x0:Q", title="Feature 0 (z-scored)"),
        y=alt.Y("x1:Q", title="Feature 1 (z-scored)"),
        color=alt.Color("cluster:N", scale=alt.Scale(scheme="tableau10"),
                        legend=alt.Legend(title="Cluster")),
        tooltip=["x0", "x1", "cluster"],
    ),
    alt.Chart(df_pred).transform_filter(~alt.datum.assigned)
    .mark_circle(size=28, opacity=0.35, color="#aaaaaa")
    .encode(x="x0:Q", y="x1:Q", tooltip=alt.value("unassigned")),
).properties(title=f"U*F predicted  (K = {clf.n_clusters_})", width=300, height=280)

alt.hconcat(true_scatter, pred_scatter)

Adjusted Rand Index (assigned samples): 0.2315


alt.HConcatChart(...)

## Appendix: threshold selection curve

U\*F picks the threshold where `regionSize(t)` has its steepest gradient — the
point where the single-origin flood first overflows into a neighbouring valley.

In [19]:
from pyesom.clustering.ustar_flood import _normalize_unit_interval, _flood_region_size

unorm = _normalize_unit_interval(clf.ustar_)
thresholds = np.linspace(0, 1, 100)
sizes = [_flood_region_size(unorm, float(t)) for t in thresholds]
grads = np.diff(sizes)

t_rule = pd.DataFrame({"t": [clf.threshold_]})

size_chart = alt.layer(
    alt.Chart(pd.DataFrame({"threshold": thresholds, "size": sizes}))
    .mark_line(color="steelblue", strokeWidth=2)
    .encode(
        x=alt.X("threshold:Q", title="threshold"),
        y=alt.Y("size:Q", title="region size (nodes)"),
    ),
    alt.Chart(t_rule).mark_rule(color="red", strokeDash=[5, 3], strokeWidth=1.5)
    .encode(x="t:Q"),
    alt.Chart(t_rule).mark_text(align="left", dx=5, dy=-8, color="red")
    .encode(x="t:Q", text=alt.value(f"t* = {clf.threshold_:.3f}")),
).properties(title="regionSize = f(threshold)", width=320, height=220)

grad_chart = alt.layer(
    alt.Chart(pd.DataFrame({"threshold": thresholds[:-1], "grad": grads}))
    .mark_line(color="darkorange", strokeWidth=2)
    .encode(
        x=alt.X("threshold:Q", title="threshold"),
        y=alt.Y("grad:Q", title="Δ region size"),
    ),
    alt.Chart(t_rule).mark_rule(color="red", strokeDash=[5, 3], strokeWidth=1.5)
    .encode(x="t:Q"),
    alt.Chart(t_rule).mark_text(align="left", dx=5, dy=-8, color="red")
    .encode(x="t:Q", text=alt.value(f"t* = {clf.threshold_:.3f}")),
).properties(title="Gradient — peak = auto threshold", width=320, height=220)

alt.hconcat(size_chart, grad_chart)

alt.HConcatChart(...)

## Summary

| Step | Function | Key output |
|---|---|---|
| U-matrix | `som.u_matrix()` | `(32, 36)` normalised distances |
| P-matrix | `compute_pmatrix(weights, data)` | `(32, 36)` density counts |
| U\*-matrix | `compute_ustar(u, p)` | `(32, 36)` density-weighted distances |
| Clustering | `UStarFloodClustering().fit(u, p)` | label grid `(32, 36)`, auto K |
| Prediction | `clf.predict(bmu_indices)` | per-sample labels `(212,)` |

Next: **03_hydrological_catchments.ipynb**.